# Structure Metrics

![Structure Metrics](https://proto-bio.github.io/proto-assets/images/tool/structure_metrics/hero.png)

This notebook demonstrates `run_structure_metrics`, which computes fast geometric quality metrics on predicted protein structures. We load a bundled example PDB, compute the longest alpha helix length and radius of gyration, and export the results for downstream filtering.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("structure_metrics")
display_overview("structure_metrics")
display_docs_section("structure_metrics", "Background")

# Structure Metrics

Structure Metrics computes coarse geometric and secondary-structure descriptors for a protein structure: secondary-structure percentages (helix, sheet, loop), the length of the longest contiguous alpha helix, and the radius of gyration. The metrics are computed using the [Biotite](https://www.biotite-python.org/) Python library and are appropriate as a fast first-pass filter for catching common structure-prediction artifacts such as unrealistically long helices or extended, non-compact conformations.

The tool assigns secondary structure using the P-SEA algorithm ([Labesse, Colloc'h, Pothier, and Mornon, 1997](https://doi.org/10.1093/bioinformatics/13.3.291)) implemented in [Biotite](https://www.biotite-python.org/), which classifies each protein residue as alpha helix, beta sheet, or loop from the Cα-atom trace alone using distance and angle patterns. The reported helix, sheet, and loop percentages summarise the overall secondary-structure composition of the structure, while the longest contiguous alpha-helix length is a separate scalar that captures the longest single helix without averaging across the structure.

Radius of gyration is the mass-weighted root-mean-square distance of all atoms from the centre of mass and is a standard scalar measure of overall structural compactness used in small-angle X-ray scattering, polymer physics, and protein structural analysis. For proteins of a given length, compact native folds produce smaller gyration radii than disordered or partially folded conformations, which is what makes the metric useful as an artifact filter for predicted structures.

Both metrics are useful as inexpensive sanity checks on structures produced by sequence-based predictors such as ESMFold, AlphaFold, Chai, Boltz, and Protenix. Predictors can default to extended helical bundles for low-confidence regions, and failed folds frequently appear as extended conformations with elevated gyration radii. The Biotite Python library ([Kunzmann et al., 2023](https://doi.org/10.1186/s12859-023-05345-6)) provides the underlying secondary-structure annotation and gyration-radius implementations used by this tool.

## Available tools

In [2]:
display_available_tools("structure_metrics")

- **`run_structure_metrics()`** — Compute structural quality metrics (SS percentages, longest helix, gyration radius) from PDB files

### `run_structure_metrics`

`run_structure_metrics` reads each PDB file with Biotite, assigns secondary structure elements via the P-SEA `annotate_sse()` algorithm (which classifies each residue as helix, sheet, or loop from the C-alpha trace alone using distance and angle patterns), and computes both the longest contiguous alpha helix (in residues) and the radius of gyration (in Angstroms). These two metrics serve as cheap filters to flag common structure-prediction artifacts — runaway single helices, and extended/disordered predictions — before running more expensive downstream analyses.

In [3]:
from pathlib import Path

from proto_tools import StructureMetricsConfig, StructureMetricsInput, StructureMetricsOutput, run_structure_metrics
from proto_tools.tools.structure_scoring.structure_metrics.structure_metrics import example_input

In [4]:
# Display input docs
display_api_reference("structure_metrics", "input", "run_structure_metrics")

# Use the tool's canonical example input — a bundled ESMFold-predicted PDB.
inputs = example_input()

**Input** — `StructureMetricsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[proto_tools.entities.structures.structure.Structure]</code> | required | Structures to compute structure metrics for |

In [5]:
# Display config docs
display_api_reference("structure_metrics", "config", "run_structure_metrics")

# StructureMetricsConfig has no tool-specific parameters; all defaults are used
config = StructureMetricsConfig()

**Config** — `StructureMetricsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the structure metrics computation
result = run_structure_metrics(inputs, config)

In [7]:
# Display output docs and inspect results
display_api_reference("structure_metrics", "output", "run_structure_metrics")

m = result.metrics[0]
print(f"Longest alpha helix: {m.longest_alpha_helix} residues")
print(f"Radius of gyration:  {m.gyration_radius:.2f} A")

**Output** — `StructureMetricsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>metrics</code> | <code>list[proto_tools.tools.structure_scoring.structure_metrics.structure_metrics.StructureQualityMetrics]</code> | <code>[]</code> | Per-structure quality metrics |

Longest alpha helix: 10 residues
Radius of gyration:  11.75 A


## Filter a Batch of Structures

A common use case is filtering a batch of predicted structures to remove artifacts. The example below applies thresholds calibrated for large globular proteins (~1000–1400 residues) — real helices rarely exceed 50 residues, and well-folded structures of this size typically have a radius of gyration under 45 A.

In [8]:
LONGEST_ALPHA_THRESHOLD = 50
GYRATION_RADIUS_THRESHOLD = 45.0

# Re-score the same Structure a few times to simulate a batch
structure = inputs.structures[0]
batch_inputs = StructureMetricsInput(structures=[structure] * 3)
batch_result = run_structure_metrics(batch_inputs, StructureMetricsConfig())

for i, m in enumerate(batch_result.metrics):
    reasons = []
    if m.longest_alpha_helix >= LONGEST_ALPHA_THRESHOLD:
        reasons.append(f"long helix ({m.longest_alpha_helix} residues)")
    if m.gyration_radius >= GYRATION_RADIUS_THRESHOLD:
        reasons.append(f"high Rg ({m.gyration_radius:.1f} A)")
    status = "FILTERED: " + ", ".join(reasons) if reasons else "PASS"
    print(f"structure[{i}]: {status}")

structure[0]: PASS
structure[1]: PASS
structure[2]: PASS


## Export Results

Structure metrics can be exported to CSV or JSON for downstream analysis or joining with other per-structure annotations.

In [9]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("structure_metrics", export_path=output_dir, file_format="csv")
print(f"Exported to {output_dir / 'structure_metrics.csv'}")

Exported to example_output/structure_metrics.csv
